# pHash 近重复算子测试

使用 `notebooks._helpers.datasets` 中的默认 MinIO sample_1000 数据集，验证单个算子 `duplicate.perceptual_duplicate_check`。

In [1]:
from pathlib import Path
import sys

# 从当前工作目录向上查找仓库根目录，保证 Notebook 从 notebooks/ 或仓库根目录启动都可用。
repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if not (repo_root / "pyproject.toml").exists():
    raise RuntimeError("cannot locate repository root")

# 优先使用当前 checkout 的源码，避免 editable install 指向其他 worktree 或旧路径。
for path in (str(repo_root / "src"), str(repo_root)):
    if path in sys.path:
        sys.path.remove(path)
for path in reversed((str(repo_root / "src"), str(repo_root))):
    sys.path.insert(0, path)

print(f"repo_root={repo_root}")

repo_root=/home/wuchaoli/codespace/ImageGallery


In [ ]:
import pandas as pd

from image_gallery.cleaning import BasicCleaner
from notebooks._helpers.datasets import load_default_minio_sample_1000_dataset, load_default_minio_sample_1000_frame
from notebooks._helpers.paths import get_notebook_library_root, reset_output_dir

# 所有清洗产物都写到 Notebook 私有目录，重复运行前会自动清理。
library_root = reset_output_dir(get_notebook_library_root("operators_phash_duplicate"))
cleaning_output_dir = library_root / "cleaning"
export_dir = library_root / "exports"
export_dir.mkdir(parents=True, exist_ok=True)

# 使用 helper 中已经准备好的 default MinIO sample_1000 数据集，不在 Notebook 内自造图片。
dataset = load_default_minio_sample_1000_dataset()
raw_frame = load_default_minio_sample_1000_frame()

print(f"library_root={library_root}")

In [ ]:
# 先展示 helper 数据集的基本形态，并验证 Dataset 对象可正常读取 image_uri。
dataset_frame = dataset.to_frame(columns=["image_id", "image_uri"])
print(f"raw_frame rows={len(raw_frame)}, columns={list(raw_frame.columns)}")
print(dataset_frame.head(10).to_string(index=False))

first_image_uri = str(dataset_frame.iloc[0]["image_uri"])
first_image_bytes = dataset.read_image_bytes(first_image_uri)
print(f"first image bytes={len(first_image_bytes)}")

assert len(dataset_frame) == 1000
assert len(first_image_bytes) > 0

In [ ]:
# 只启用单个 pHash 近重复算子，使用默认 max_distance=10，避免其他清洗规则干扰验证结果。
operator_configs = [{"duplicate.perceptual_duplicate_check": {}}]
cleaner = BasicCleaner(operator_configs).compile()

# 编译计划必须先计算 phash，再做 dataset_aggregate 级别的近重复分组。
plan_frame = cleaner.plan()
print(plan_frame.to_string(index=False))
assert plan_frame["computer_name"].tolist() == [
    "image_perceptual_hash_computer",
    "perceptual_duplicate_group_computer",
]

# 执行后会写出 parameter_table、evaluation_table、state 和 relation table。
cleaner.run(dataset, output_dir=cleaning_output_dir)
run_dirs = sorted(path for path in cleaning_output_dir.iterdir() if path.is_dir())
assert len(run_dirs) == 1
run_dir = run_dirs[0]
print(f"run_dir={run_dir}")

In [ ]:
# 读取运行产物，分别检查参数、评估结果和近重复 pair relation。
parameter_table = pd.read_parquet(run_dir / "parameter_table.parquet")
evaluation_table = pd.read_parquet(run_dir / "evaluation_table.parquet")
relation_table = pd.read_parquet(run_dir / "relations" / "perceptual_duplicate_pairs.parquet")

print("parameter_table")
print(
    parameter_table[
        [
            "image_id",
            "phash",
            "perceptual_duplicate_group_id",
            "perceptual_duplicate_count",
            "perceptual_duplicate_distance",
        ]
    ].head(20).to_string(index=False)
)
print("evaluation_table")
print(
    evaluation_table[
        [
            "image_id",
            "perceptual_duplicate_action",
            "perceptual_duplicate_reason",
            "final_action",
        ]
    ].head(20).to_string(index=False)
)
print("relation_table")
print(relation_table.head(20).to_string(index=False))
print("action counts")
print(evaluation_table["perceptual_duplicate_action"].value_counts(dropna=False).to_string())

In [ ]:
# 导出 clean/drop 表，便于在 Notebook 中单独检查保留与删除结果。
clean_table = cleaner.export("clean", str(export_dir / "clean.parquet")).to_frame()
drop_table = cleaner.export("dropped", str(export_dir / "dropped.parquet")).to_frame()

print(f"clean_export_path={export_dir / 'clean.parquet'}")
print(f"drop_export_path={export_dir / 'dropped.parquet'}")
print(f"clean rows={len(clean_table)}, drop rows={len(drop_table)}")

print("clean_table")
print(
    clean_table[
        [
            "image_id",
            "perceptual_duplicate_action",
            "perceptual_duplicate_reason",
            "final_action",
        ]
    ].head(20).to_string(index=False)
)
print("drop_table")
print(
    drop_table[
        [
            "image_id",
            "perceptual_duplicate_action",
            "perceptual_duplicate_reason",
            "final_action",
        ]
    ].head(20).to_string(index=False)
)


In [ ]:
# 生成静态 HTML 预览页，重点检查 pHash drop 结果的分组效果。
preview_path = cleaner.preview_html(
    library_root / "preview.html",
    action="drop",
    groupby="perceptual_duplicate_group_id",
    include_group_context=True,
    sort_by=["perceptual_duplicate_count", "perceptual_duplicate_distance"],
    ascending=[False, True],
    caption_columns=[
        "image_id",
        "final_action",
        "perceptual_duplicate_distance",
        "perceptual_duplicate_reason",
    ],
    max_groups=20,
    max_items_per_group=12,
    thumbnail_size=160,
)

print(f"preview_html_path={preview_path}")
assert preview_path.exists()


In [ ]:
# 参数断言：helper 数据集完整跑完，pHash 参数和近重复参数都被写入。
parameter_rows = parameter_table.set_index("image_id")
evaluation_rows = evaluation_table.set_index("image_id")

assert len(parameter_table) == 1000
assert len(evaluation_table) == 1000
assert parameter_table["phash"].fillna("").astype(str).str.len().eq(16).any()
assert {
    "perceptual_duplicate_group_id",
    "perceptual_duplicate_count",
    "perceptual_duplicate_distance",
}.issubset(parameter_table.columns)
assert {
    "perceptual_duplicate_action",
    "perceptual_duplicate_reason",
    "final_action",
}.issubset(evaluation_table.columns)

# 真实数据集不强行要求一定存在近重复；若有 relation 命中，则校验 relation 与 drop action 一致。
expected_relation_columns = {
    "relation_type",
    "source_image_id",
    "target_image_id",
    "score",
    "group_id",
    "parameter_name",
    "computer_name",
}
assert expected_relation_columns.issubset(relation_table.columns)
assert len(clean_table) + len(drop_table) == len(evaluation_table)
assert set(clean_table["final_action"]) == {"keep"}
assert set(drop_table["final_action"]) == {"drop"}
assert (export_dir / "clean.parquet").exists()
assert (export_dir / "dropped.parquet").exists()
assert (library_root / "preview.html").exists()
if not relation_table.empty:
    assert set(relation_table["relation_type"]) == {"perceptual_duplicate"}
    duplicate_targets = set(relation_table["target_image_id"].astype(str))
    target_actions = evaluation_rows.loc[list(duplicate_targets), "perceptual_duplicate_action"]
    assert set(target_actions) == {"drop"}
    target_distances = parameter_rows.loc[list(duplicate_targets), "perceptual_duplicate_distance"]
    assert pd.to_numeric(target_distances, errors="coerce").le(10).all()

print("PASS: phash duplicate operator notebook smoke completed")